# Arrhenius Kinetics Exploration

This notebook explores the Arrhenius degradation model that underlies ColdGuard's potency estimation.

## Overview
- Visualize how degradation rate varies with temperature
- Compare degradation kinetics across vaccine types
- Illustrate the trapezoidal integration approach


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from core.arrhenius import arrhenius_k, integrate_degradation, compute_potency
from core.vaccine_params import VACCINE_DB

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Rate Constant vs Temperature

The Arrhenius equation: `k(T) = A * exp(-Ea / (R * T))`

This shows how the degradation rate constant increases exponentially with temperature.

In [ ]:
temps_C = np.linspace(-20, 40, 200)
temps_K = temps_C + 273.15

fig, ax = plt.subplots()
for key, vp in VACCINE_DB.items():
    k_vals = [arrhenius_k(T, vp.Ea_mean, vp.A) for T in temps_K]
    ax.semilogy(temps_C, k_vals, label=key)

ax.axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='5°C (storage)')
ax.axvline(x=25, color='red', linestyle='--', alpha=0.5, label='25°C (room temp)')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Degradation Rate k (hr⁻¹)')
ax.set_title('Arrhenius Degradation Rate vs Temperature')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Potency Over Time at Constant Temperatures

First-order degradation: `P(t) = exp(-k(T) * t)`

In [ ]:
hours = np.arange(0, 169)  # 7 days
storage_temps_C = [4.0, 8.0, 15.0, 25.0]
vaccine = 'DPT'
vp = VACCINE_DB[vaccine]

fig, ax = plt.subplots()
for tc in storage_temps_C:
    timestamps = list(hours * 3600.0)
    temperatures = [tc] * len(hours)
    potencies = []
    for h in hours:
        ts_sub = list(hours[:h+1] * 3600.0)
        tc_sub = [tc] * (h+1)
        potencies.append(compute_potency(ts_sub, tc_sub, vp.Ea_mean, vp.A))
    ax.plot(hours, [p*100 for p in potencies], label=f'{tc}°C')

ax.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% threshold')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Remaining Potency (%)')
ax.set_title(f'{vaccine} Potency Over 7 Days at Constant Temperatures')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Trapezoidal Integration Illustration

ColdGuard uses trapezoidal integration to compute cumulative degradation over a temperature trajectory.

In [ ]:
# Simulate a cold chain excursion scenario
rng = np.random.default_rng(42)
n_hours = 48
base_temp = 4.0 + rng.normal(0, 0.3, n_hours + 1)

# Add excursion hours 20-28
base_temp[20:28] = 20.0 + rng.normal(0, 0.5, 8)

hours = np.arange(n_hours + 1)
timestamps = (hours * 3600).tolist()
vp = VACCINE_DB['DPT']

# Compute rate at each point
k_vals = [arrhenius_k(tc + 273.15, vp.Ea_mean, vp.A) for tc in base_temp]

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

ax1.plot(hours, base_temp, 'b-', linewidth=2)
ax1.axhline(y=8, color='red', linestyle='--', alpha=0.5, label='8°C threshold')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Temperature Trajectory and Degradation Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.fill_between(hours, k_vals, alpha=0.4, color='orange', label='k(T) — rate')
ax2.plot(hours, k_vals, 'orange', linewidth=2)
ax2.set_xlabel('Time (hours)')
ax2.set_ylabel('Rate k(T) (hr⁻¹)')
ax2.set_title('Degradation Rate (area = cumulative degradation D)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

potency = compute_potency(timestamps, base_temp.tolist(), vp.Ea_mean, vp.A)
print(f'Remaining potency: {potency*100:.2f}%')

## 4. Comparing All Vaccines at 25°C

OPV is the most temperature-sensitive vaccine; HepB is the most stable.

In [ ]:
hours = np.arange(0, 169)  # 7 days
tc = 25.0  # room temperature

fig, ax = plt.subplots()
for key, vp in VACCINE_DB.items():
    potencies = []
    for h in hours:
        ts_sub = list(hours[:h+1] * 3600.0)
        tc_sub = [tc] * (h+1)
        potencies.append(compute_potency(ts_sub, tc_sub, vp.Ea_mean, vp.A))
    ax.plot(hours, [p*100 for p in potencies], label=key)

ax.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% threshold')
ax.axhline(y=67, color='orange', linestyle='--', alpha=0.7, label='67% threshold (OPV)')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Remaining Potency (%)')
ax.set_title(f'All Vaccines at {tc}°C — Potency Over 7 Days')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()